In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
titanic = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

titanic['Title'] = titanic['Name'].str.extract(r',\s*([^.]*)\.')
combined_median = titanic.groupby(['Title', 'Pclass'])['Age'].median()
title_median = titanic.groupby('Title')['Age'].median()
titanic['Age_filled_combined'] = titanic.groupby(['Title', 'Pclass'])['Age'].transform('median')
titanic['Age_filled_title'] = titanic.groupby('Title')['Age'].transform('median')
titanic['Age'] = titanic['Age'].fillna(titanic['Age_filled_combined']).fillna(titanic['Age_filled_title'])

titanic.drop(columns=['Age_filled_combined','Age_filled_title'],inplace=True)

overall_survival_rate = titanic['Survived'].mean()

top_class = titanic.groupby('Pclass')['Survived'].mean().idxmax().astype(str)
survival_by_class = titanic.groupby('Pclass')['Survived'].mean()

import matplotlib.pyplot as plt

plt.bar(survival_by_class.index,survival_by_class.values)
plt.xlabel('P Class')
plt.ylabel('Survival Rate')
plt.show()

survival_by_sex = titanic.groupby('Sex')['Survived'].mean().to_dict()

from statsmodels.stats.proportion import proportions_ztest

success=titanic.groupby('Sex')['Survived'].sum()
total_count=titanic.groupby('Sex')['Survived'].count()

z_score, p_value = proportions_ztest(count=success, nobs=total_count, alternative='two-sided')
sex_survival_pvalue = p_value

alpha = 0.05
if sex_survival_pvalue < alpha:
    sex_survival_result = "reject"
else : 
    sex_survival_result = "fail to reject"

print(sex_survival_result)

max_age=int(np.ceil(titanic['Age'].max()/10)) * 10
age_bins = list(range(0,max_age+5,5))
titanic['Age Group']=pd.cut(titanic['Age'],bins=age_bins, labels=None)
age_group_survival = titanic.groupby('Age Group')['Survived'].mean()

plt.bar(age_group_survival.index.astype(str), age_group_survival.values)
plt.xticks(rotation=90)
plt.xlabel('Age Groups')
plt.ylabel('Survival Rate')
plt.show()

titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch']

titanic['Sex_male'] = (titanic['Sex'] == 'male').astype(int)

features = ['Pclass', 'Sex_male', 'Age', 'FamilySize']
X = titanic[features]
y = titanic['Survived']

import statsmodels.api as sm

X_const = sm.add_constant(X)
logit_model = sm.Logit(y,X_const).fit()

titanic_test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
titanic_test['Title'] = titanic_test['Name'].str.extract(r',\s*([^.]*)\.')

titanic_test['Age_filled_combined'] = titanic_test.set_index(['Title', 'Pclass']).index.map(combined_median)
titanic_test['Age_filled_title'] = titanic_test['Title'].map(title_median)
titanic_test['Age'] = titanic_test['Age'].fillna(titanic_test['Age_filled_combined']).fillna(titanic_test['Age_filled_title'])

titanic_test.drop(columns=['Age_filled_combined','Age_filled_title'],inplace=True)
titanic_test.isna().sum()

titanic_test['FamilySize'] = titanic_test['SibSp'] + titanic_test['Parch']

titanic_test['Sex_male'] = (titanic_test['Sex'] == 'male').astype(int)

features = ['Pclass', 'Sex_male', 'Age', 'FamilySize']
X_test = titanic_test[features]
X_test_with_const = sm.add_constant(X_test, has_constant='add')

prediction = logit_model.predict(X_test_with_const)

titanic_test['Survived'] = (prediction >= 0.5).astype(int)